# 04_index_sample_to_solr

MongoDB から少件数の JaLC 文書を取り出し、Solr 登録用データを確認してから sample 登録する。

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# 設定と登録処理関数を読み込む
from pprint import pprint

from tanigawa_shoshi.config import BATCH_SIZE, SOLR_BASE_URL, SOLR_CORE
from tanigawa_shoshi.solr_indexer import (
    build_documents,
    delete_all_documents,
    get_mongo_collection,
    get_solr_client,
    index_documents,
    iter_jalc_documents,
)


In [ ]:
# 少件数だけ MongoDB から取得する
sample_size = 5

collection = get_mongo_collection()
sample_raw_docs = list(iter_jalc_documents(collection, limit=sample_size))

print(f"sample raw docs: {len(sample_raw_docs)}")


In [ ]:
# Solr 登録用データへ変換した結果を確認する
sample_solr_docs, build_stats = build_documents(sample_raw_docs)

print("build stats")
pprint(build_stats)
print()

if sample_solr_docs:
    print("first solr document")
    pprint(sample_solr_docs[0])
else:
    print("登録対象文書がありません。")


In [ ]:
# sample 文書を Solr に登録する
# 実際に登録する時だけこのセルを実行する
solr = get_solr_client(SOLR_BASE_URL, SOLR_CORE)
index_stats = index_documents(solr, sample_solr_docs, batch_size=BATCH_SIZE)

print("index stats")
pprint(index_stats)


In [ ]:
# Solr コアは残したまま、登録済み文書だけを全削除する
# 実際に削除する時だけこのセルを実行する
solr = get_solr_client(SOLR_BASE_URL, SOLR_CORE)
delete_stats = delete_all_documents(solr)

print("delete stats")
pprint(delete_stats)
